# Web-Gold-40K — time-safe v2.7 epoch-4 continuation

Use this notebook only with two Kaggle inputs: (1) the original Web-Gold-40K dataset and (2) the failed v2.7 notebook output containing the completed epoch-3 checkpoint directory, the four-row metrics CSV, and the environment JSON. It restores that checkpoint's weights, optimizer, scheduler, and scaler; trains only missing epoch 4; then selects and verifies one checkpoint across epochs 0–4. It never reads the test split and never combines rows from independent runs.

The historical checkpoint did not store RNG state. The report therefore marks epoch 4 as a deterministic registered-seed continuation, not a bit-for-bit replay of the interrupted in-memory epoch-4 stream.

In [1]:
# 1. Pull the modular implementation and reproduce the original package environment.
from pathlib import Path
import csv
import importlib.metadata as metadata
import json, os, subprocess, sys
import torch

assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before Run All.'
GPU_NAME = torch.cuda.get_device_name(0)
assert 'T4' in GPU_NAME.upper(), f'This continuation requires a T4, but Kaggle assigned {GPU_NAME}. Stop the session, select a T4 accelerator, and restart.'
print('Verified GPU:', GPU_NAME)

REPOSITORY = 'https://github.com/Kiyas-Mahmud/webagent.git'
REPO_ROOT = Path('/kaggle/working/webagent')
SOURCE_ROOT = REPO_ROOT / 'src'
if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'Code'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'Code', '--single-branch', REPOSITORY, str(REPO_ROOT)], check=True)
requirements = [
    'transformers==4.57.6', 'peft==0.19.1', 'bitsandbytes==0.49.2',
    'accelerate==1.14.0', 'scikit-learn==1.9.0',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', *requirements], check=True)
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
os.chdir(REPO_ROOT)
RESUME_GIT_COMMIT = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
current_environment = {name: metadata.version(name) for name in ['torch', 'transformers', 'peft', 'bitsandbytes', 'accelerate', 'scikit-learn']}
current_environment['python'] = sys.version.split()[0]
current_environment['resume_git_commit'] = RESUME_GIT_COMMIT
RESUME_ENVIRONMENT_PATH = Path('/kaggle/working/gold_recovery_v2_7_resume_environment.json')
print(json.dumps(current_environment, indent=2))

Verified GPU: Tesla T4


Cloning into '/kaggle/working/webagent'...


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.3 MB/s eta 0:00:00
{
  "torch": "2.10.0+cu128",
  "transformers": "4.57.6",
  "peft": "0.19.1",
  "bitsandbytes": "0.49.2",
  "accelerate": "1.14.0",
  "scikit-learn": "1.9.0",
  "python": "3.12.13",
  "resume_git_commit": "3b231967573022210a84a652e2fc0e046ab03513"
}


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.


In [2]:
# 2. Locate exactly one dataset and exactly one interrupted-run lineage.
# Prior Kaggle outputs may be mounted as loose files or as results.zip.
import shutil
import zipfile
from pathlib import PurePosixPath

INPUT_ROOT = Path('/kaggle/input')
ATTACHED_DATA_ROOT = Path('/kaggle/input/datasets/kiyasmahmud/web-gold-40k')
SPLIT_FILES = ('split_train.json', 'split_val.json', 'split_test.json')

def find_split_root(root: Path) -> Path:
    if all((root / name).is_file() for name in SPLIT_FILES):
        return root
    candidates = []
    for current, _, files in os.walk(root, followlinks=True):
        if set(SPLIT_FILES).issubset(files):
            candidates.append(Path(current))
    if len(candidates) != 1:
        raise FileNotFoundError(f'Expected one structured split folder; found {candidates}')
    return candidates[0]

def require_one_loose(pattern: str) -> Path | None:
    matches = sorted(INPUT_ROOT.rglob(pattern))
    if len(matches) > 1:
        raise AssertionError(f'Attach only one prior output; multiple {pattern} files found: {matches}')
    return matches[0] if matches else None

def locate_prior_archive() -> tuple[Path, dict[str, str]]:
    required = {'gold_mini_recovery_v2_7_metrics.csv', 'gold_recovery_v2_7_environment.json'}
    candidates = []
    for archive_path in sorted(INPUT_ROOT.rglob('*.zip')):
        try:
            with zipfile.ZipFile(archive_path) as archive:
                members_by_name = {}
                for member in archive.namelist():
                    members_by_name.setdefault(PurePosixPath(member).name, []).append(member)
                checkpoint_names = {
                    epoch: [name for name in members_by_name if name.startswith(f'best_e{epoch}_') and name.endswith('.ckpt')]
                    for epoch in range(4)
                }
                epoch_four = [name for name in members_by_name if name.startswith('best_e4_') and name.endswith('.ckpt')]
                unique_required = required.issubset(members_by_name) and all(len(members_by_name[name]) == 1 for name in required)
                unique_checkpoints = all(
                    len(checkpoint_names[epoch]) == 1 and len(members_by_name[checkpoint_names[epoch][0]]) == 1
                    for epoch in range(4)
                )
                if unique_required and unique_checkpoints and not epoch_four:
                    selected = {name: members_by_name[name][0] for name in required}
                    for epoch in range(4):
                        basename = checkpoint_names[epoch][0]
                        selected[basename] = members_by_name[basename][0]
                    candidates.append((archive_path, selected))
        except zipfile.BadZipFile:
            continue
    if len(candidates) != 1:
        raise AssertionError(
            'Attach exactly one prior v2.7 output, either as loose files or a ZIP. '
            f'Compatible ZIP archives found: {[path for path, _ in candidates]}'
        )
    return candidates[0]

def extract_prior_artifacts(archive_path: Path, members: dict[str, str]) -> Path:
    destination = Path('/kaggle/working/v2_7_prior_artifacts')
    checkpoint_destination = destination / 'checkpoints'
    checkpoint_destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for basename, member in members.items():
            target = checkpoint_destination / basename if basename.endswith('.ckpt') else destination / basename
            info = archive.getinfo(member)
            if target.is_file() and target.stat().st_size == info.file_size:
                continue
            print(f'Extracting {basename} ({info.file_size / 1e6:.1f} MB) ...')
            with archive.open(info) as source, target.open('wb') as output:
                shutil.copyfileobj(source, output, length=8 * 1024 * 1024)
            assert target.stat().st_size == info.file_size
    return destination

DATA_ROOT = find_split_root(ATTACHED_DATA_ROOT).resolve()
PRIOR_METRICS = require_one_loose('gold_mini_recovery_v2_7_metrics.csv')
if PRIOR_METRICS is not None:
    PRIOR_ENVIRONMENT = require_one_loose('gold_recovery_v2_7_environment.json')
    RESUME_CHECKPOINT = require_one_loose('best_e3_outcome-mcc*.ckpt')
    assert PRIOR_ENVIRONMENT is not None and RESUME_CHECKPOINT is not None, 'Loose prior output is incomplete.'
    PRIOR_SOURCE = 'loose Kaggle input files'
else:
    PRIOR_ARCHIVE, PRIOR_MEMBERS = locate_prior_archive()
    EXTRACTED_PRIOR_ROOT = extract_prior_artifacts(PRIOR_ARCHIVE, PRIOR_MEMBERS)
    PRIOR_METRICS = EXTRACTED_PRIOR_ROOT / 'gold_mini_recovery_v2_7_metrics.csv'
    PRIOR_ENVIRONMENT = EXTRACTED_PRIOR_ROOT / 'gold_recovery_v2_7_environment.json'
    RESUME_CHECKPOINT = next((EXTRACTED_PRIOR_ROOT / 'checkpoints').glob('best_e3_outcome-mcc*.ckpt'))
    PRIOR_SOURCE = f'ZIP archive: {PRIOR_ARCHIVE}'
EXPECTED_RESUME_SHA256 = '2f4a7e415c6382f0983d2c705d2cd6526d5e766c42319ff40f2adcc71240f9ba'
PRIOR_CHECKPOINT_DIR = RESUME_CHECKPOINT.parent
for epoch in range(4):
    matches = list(PRIOR_CHECKPOINT_DIR.glob(f'best_e{epoch}_*.ckpt'))
    assert len(matches) == 1, f'Expected one completed epoch-{epoch} checkpoint: {matches}'
assert not list(PRIOR_CHECKPOINT_DIR.glob('best_e4_*.ckpt')), 'Input already contains a completed epoch-4 checkpoint; do not resume it.'
original_environment = json.loads(PRIOR_ENVIRONMENT.read_text(encoding='utf-8'))
for name in ['torch', 'transformers', 'peft', 'bitsandbytes', 'accelerate', 'scikit-learn', 'python']:
    assert current_environment[name] == original_environment[name], f'Environment mismatch for {name}: {current_environment[name]} != {original_environment[name]}'
assert original_environment['git_commit'] == 'c583e24fdfddac417e84e8cf306b251c305988fe', original_environment['git_commit']
current_environment['original_training_environment'] = original_environment
RESUME_ENVIRONMENT_PATH.write_text(json.dumps(current_environment, indent=2), encoding='utf-8')
print('DATA_ROOT:', DATA_ROOT)
print('prior source:', PRIOR_SOURCE)
print('prior metrics:', PRIOR_METRICS)
print('resume checkpoint:', RESUME_CHECKPOINT)
print('original commit:', original_environment['git_commit'])
print('resume code commit:', RESUME_GIT_COMMIT)

DATA_ROOT: /kaggle/input/datasets/kiyasmahmud/web-gold-40k/final_data_set_40k
prior source: loose Kaggle input files
prior metrics: /kaggle/input/datasets/kiyasmahmud/result-2-7/webagent/results/gold_mini_recovery_v2_7_metrics.csv
resume checkpoint: /kaggle/input/datasets/kiyasmahmud/result-2-7/webagent/checkpoints/Y_QWEN2VL_2B_GOLD_V2_7_MINI_RECOVERY_V2_7/best_e3_outcome-mcc0.562.ckpt
original commit: c583e24fdfddac417e84e8cf306b251c305988fe
resume code commit: 3b231967573022210a84a652e2fc0e046ab03513


In [3]:
# 3. Re-run the train/validation-only geometry audit before allocating the model.
from web_agent.config import load_config
from web_agent.train.gold_stages import run_gold_bbox_audit

audit_cfg = load_config('configs/backbones/qwen2vl_2b_gold_v2_7.yaml')
audit_cfg['data']['root'] = str(DATA_ROOT)
audit_cfg['data']['num_workers'] = 0
bbox_audit = run_gold_bbox_audit(audit_cfg)
AUDIT_PATH = Path('/kaggle/working/gold_recovery_v2_7_resume_bbox_audit.json')
AUDIT_PATH.write_text(json.dumps(bbox_audit, indent=2), encoding='utf-8')
assert bbox_audit['status'] == 'PASS', bbox_audit
assert bbox_audit['test_rows_read'] == 0
print('BBox audit passed:', AUDIT_PATH)

BBox audit passed: /kaggle/working/gold_recovery_v2_7_resume_bbox_audit.json


In [4]:
# 4. Restore epoch 3 and train only epoch 4 in an isolated process.
REPORT_PATH = Path('/kaggle/working/gold_recovery_v2_7_resume_report.json')
CSV_PATH = Path('/kaggle/working/gold_recovery_v2_7_resume_result.csv')
DIAGNOSTICS_PATH = Path('/kaggle/working/gold_recovery_v2_7_resume_diagnostics.json')
RESUMED_METRICS_PATH = Path('/kaggle/working/gold_recovery_v2_7_resumed_epoch_metrics.csv')
SELECTED_CHECKPOINT = Path('/kaggle/working/gold_recovery_v2_7_selected.ckpt')
CHECKPOINT_OUTPUT_ROOT = Path('/kaggle/working/checkpoints')
command = [
    sys.executable, str(REPO_ROOT / 'scripts' / 'resume_v2_7_mini.py'),
    '--config', 'configs/backbones/qwen2vl_2b_gold_v2_7.yaml',
    '--data-root', str(DATA_ROOT),
    '--prior-metrics-csv', str(PRIOR_METRICS),
    '--prior-checkpoint-dir', str(PRIOR_CHECKPOINT_DIR),
    '--resume-checkpoint', str(RESUME_CHECKPOINT),
    '--expected-resume-sha256', EXPECTED_RESUME_SHA256,
    '--selected-checkpoint', str(SELECTED_CHECKPOINT),
    '--report', str(REPORT_PATH), '--csv', str(CSV_PATH),
    '--diagnostics-json', str(DIAGNOSTICS_PATH),
    '--resumed-metrics-csv', str(RESUMED_METRICS_PATH),
    '--checkpoint-output-root', str(CHECKPOINT_OUTPUT_ROOT),
    '--train-rows', '5000', '--val-rows', '500',
    '--epochs', '5', '--resume-epoch', '3', '--seed', '42',
    '--num-workers', '4', '--min-pixels', '50176', '--max-pixels', '200704',
]
print('Running controlled epoch-4 continuation...')
subprocess.run(command, check=True)

Running controlled epoch-4 continuation...


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.44s/it]


trainable params: 18,464,768 || all params: 2,227,450,368 || trainable%: 0.8290
train rows: 5000
action weights: [1.01, 1.01, 1.02, 1.01, 0.88, 1.1]
failure weights: [0.57, 0.97, 0.9, 8.33]
outcome weights: [1.3, 1.0]
recovery weights: [0.0, 0.0, 0.83, 1.49, 0.67, 0.0]
recovery pos_weight: 1.93 (attempted success=407, failure=784)
needs-recovery pos_weight: 3.2 (needed=1191, not-needed=3809)
bbox log-size prior: {'source': 'valid training bbox rows only', 'valid_rows': 1414, 'excluded_invalid_rows': 448, 'log_wh': [-1.9972667424598038, -3.037358655551862], 'geometric_mean_wh': [0.1357056954052972, 0.04796140492585266]}
epoch 4 | step 50/157 | loss -1.937 | elapsed 41.4min | ETA 88.5min
epoch 4 | step 100/157 | loss -2.409 | elapsed 82.5min | ETA 47.0min
epoch 4 | step 150/157 | loss -2.111 | elapsed 123.7min | ETA 5.8min
epoch 4: train_loss=-1.9067  failure_f1=0.8262  failure_macro_f1=0.7926  outcome_bal_acc=0.7932  outcome_mcc=0.5852  success_recall=0.7644  outcome_brier=0.1583  outco

CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/webagent/scripts/resume_v2_7_mini.py', '--config', 'configs/backbones/qwen2vl_2b_gold_v2_7.yaml', '--data-root', '/kaggle/input/datasets/kiyasmahmud/web-gold-40k/final_data_set_40k', '--prior-metrics-csv', '/kaggle/input/datasets/kiyasmahmud/result-2-7/webagent/results/gold_mini_recovery_v2_7_metrics.csv', '--prior-checkpoint-dir', '/kaggle/input/datasets/kiyasmahmud/result-2-7/webagent/checkpoints/Y_QWEN2VL_2B_GOLD_V2_7_MINI_RECOVERY_V2_7', '--resume-checkpoint', '/kaggle/input/datasets/kiyasmahmud/result-2-7/webagent/checkpoints/Y_QWEN2VL_2B_GOLD_V2_7_MINI_RECOVERY_V2_7/best_e3_outcome-mcc0.562.ckpt', '--expected-resume-sha256', '2f4a7e415c6382f0983d2c705d2cd6526d5e766c42319ff40f2adcc71240f9ba', '--selected-checkpoint', '/kaggle/working/gold_recovery_v2_7_selected.ckpt', '--report', '/kaggle/working/gold_recovery_v2_7_resume_report.json', '--csv', '/kaggle/working/gold_recovery_v2_7_resume_result.csv', '--diagnostics-json', 

In [5]:
# 5. Enforce the complete five-epoch, selection, provenance, and export contract.
report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
quality = report['quality_gates']
provenance = report['resume_provenance']
assert report['status'] == 'PASS' and quality['status'] == 'PASS'
assert report['test_rows_read'] == 0
assert [int(row['epoch']) for row in report['history']] == [0, 1, 2, 3, 4]
assert provenance['prior_epochs'] == [0, 1, 2, 3]
assert provenance['resumed_epochs'] == [4]
assert provenance['same_checkpoint_lineage'] is True
assert provenance['independent_run_rows_combined'] is False
assert provenance['optimizer_restored'] is True
assert provenance['scheduler_restored'] is True
assert provenance['scaler_restored'] is True
assert provenance['rng_state_restored'] is False
assert provenance['bitwise_stochastic_continuation'] is False
assert report['checkpoint_roundtrip'] is True
assert report['selected_checkpoint_revalidation']['status'] == 'PASS'
assert quality['eligible_epochs']
selected_epoch = int(report['selected_epoch'])
assert quality['selected_epoch'] == selected_epoch
assert quality['selected_epoch_is_eligible'] is True
assert report['best_checkpoint'] == str(SELECTED_CHECKPOINT)
assert quality['selected_checkpoint'] == report['best_checkpoint']
assert SELECTED_CHECKPOINT.is_file()
assert all(path.is_file() for path in [REPORT_PATH, CSV_PATH, DIAGNOSTICS_PATH, RESUMED_METRICS_PATH, AUDIT_PATH, RESUME_ENVIRONMENT_PATH])
with CSV_PATH.open(encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
assert [int(row['epoch']) for row in rows] == [0, 1, 2, 3, 4]
assert len([row for row in rows if row['is_selected'] == 'True']) == 1
print('V2.7 FIVE-EPOCH CONTINUATION PASSED')
print('eligible epochs:', quality['eligible_epochs'])
print('selected epoch:', selected_epoch, '| outcome MCC:', report['best_metric'])
print('RNG limitation:', provenance['limitation'])
print('report:', REPORT_PATH)
print('CSV:', CSV_PATH)
print('diagnostics:', DIAGNOSTICS_PATH)
print('selected checkpoint:', SELECTED_CHECKPOINT)

V2.7 FIVE-EPOCH CONTINUATION PASSED
eligible epochs: [3]
selected epoch: 3 | outcome MCC: 0.56157
RNG limitation: The historical epoch-3 checkpoint predates RNG-state saving; epoch 4 uses the registered seed but is not a bitwise replay of the interrupted in-memory epoch-4 stream.
report: /kaggle/working/gold_recovery_v2_7_resume_report.json
CSV: /kaggle/working/gold_recovery_v2_7_resume_result.csv
diagnostics: /kaggle/working/gold_recovery_v2_7_resume_diagnostics.json
selected checkpoint: /kaggle/working/gold_recovery_v2_7_selected.ckpt


## What PASS means

All five epoch rows share one checkpoint lineage: epochs 0–3 come from the attached interrupted run, epoch 4 starts from its completed epoch-3 checkpoint, and the registered all-gates-then-outcome-MCC selector is applied only after completion. The selected physical checkpoint is reloaded, perturbed/restored for a round-trip test, and re-evaluated on the same locked 500-row validation subset. Preserve every `/kaggle/working/gold_recovery_v2_7_*` artifact.